# 01 — Exploration audio : waveform, STFT, spectrogrammes

Objectifs :
- Charger un fichier audio et visualiser sa waveform
- Calculer une STFT, afficher le spectrogramme en dB
- Vérifier la reconstruction (round-trip STFT → ISTFT)
- Tester le `PairedAudioDataset` PyTorch et un `DataLoader`

## ⚠️ Setup — toujours exécuter cette cellule en premier

Chaque notebook Colab a son **propre kernel** : le `sys.path` de `main.ipynb` n'est pas partagé.
Cette cellule ajoute le repo au `sys.path` pour que `from src import ...` fonctionne.

In [ ]:
import os, sys
REPO_DIR = '/content/Filtre-Voix-DL'
assert os.path.exists(REPO_DIR), (
    f'{REPO_DIR} introuvable — exécute la cellule clone de main.ipynb '
    'pour cloner/mettre à jour le repo sur ce runtime Colab.'
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception as e:
    pass  # bug connu Colab Python 3.12, sans impact

print(f'sys.path OK — repo : {REPO_DIR}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src import config
from src import audio as A
from src.dataset import list_pairs, PairedAudioDataset

## 1. Charger une paire (noisy / clean)

In [ ]:
# pair_by='auto' : essaie par nom, bascule sur index si les noms ne correspondent pas
pairs = list_pairs(pair_by='auto')
assert pairs, 'Aucune paire trouvée — vérifie que data/clean et data/noisy sont remplis.'
print(f'{len(pairs)} paires disponibles')

label, noisy_path, clean_path = pairs[0]
print(f'Exemple : {label}')
print(f'  noisy : {noisy_path}')
print(f'  clean : {clean_path}')

noisy = A.load_audio(noisy_path)
clean = A.load_audio(clean_path)
print(f'noisy : shape={noisy.shape} | durée={noisy.shape[-1] / config.SAMPLE_RATE:.2f}s')
print(f'clean : shape={clean.shape} | durée={clean.shape[-1] / config.SAMPLE_RATE:.2f}s')

In [ ]:
print('Noisy :'); display(Audio(noisy, rate=config.SAMPLE_RATE))
print('Clean :'); display(Audio(clean, rate=config.SAMPLE_RATE))

## 2. Waveform

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
A.plot_waveform(noisy, title='Noisy', ax=axes[0])
A.plot_waveform(clean, title='Clean', ax=axes[1])
plt.tight_layout(); plt.show()

## 3. Spectrogrammes (STFT en dB)

In [ ]:
print(f'STFT params : n_fft={config.N_FFT}, hop={config.HOP_LENGTH}, win={config.WIN_LENGTH}')
A.plot_pair(noisy, clean)
plt.show()

## 4. Round-trip STFT → ISTFT

Vérifie que STFT puis ISTFT redonnent (à epsilon près) le signal original.
Indispensable avant d'attaquer le débruitage.

In [ ]:
wav = A.fix_length(clean, config.CLIP_SAMPLES, mode='center')
spec = A.stft(wav)
recon = A.istft(spec, length=wav.shape[-1])

err = np.abs(wav - recon).max()
rms = np.sqrt(np.mean((wav - recon) ** 2))
print(f'spec shape  : {spec.shape}  (freq, time)')
print(f'wav  shape  : {wav.shape}')
print(f'erreur max  : {err:.2e}')
print(f'erreur RMS  : {rms:.2e}')
assert err < 1e-3, 'Reconstruction trop éloignée — vérifier les params STFT.'

## 5. Test du `PairedAudioDataset` + `DataLoader`

In [ ]:
from torch.utils.data import DataLoader

ds = PairedAudioDataset(return_spectrogram=True, crop_mode='random', pair_by='auto')
print(f'Taille dataset : {len(ds)}')

sample = ds[0]
for k, v in sample.items():
    print(f'  {k:14s} : {type(v).__name__}', getattr(v, 'shape', v))

In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)
batch = next(iter(loader))
print('Batch :')
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f'  {k:14s} : shape={tuple(v.shape)} | dtype={v.dtype}')
    else:
        print(f'  {k:14s} : {v}')

In [ ]:
import librosa
mag = batch['noisy_mag'][0, 0].numpy()
mag_clean = batch['clean_mag'][0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, m, title in zip(axes, [mag, mag_clean], ['Noisy mag', 'Clean mag']):
    db = librosa.amplitude_to_db(m, ref=np.max)
    img = librosa.display.specshow(db, sr=config.SAMPLE_RATE,
                                   hop_length=config.HOP_LENGTH,
                                   x_axis='time', y_axis='hz', ax=ax)
    ax.set_title(title)
    plt.colorbar(img, ax=ax, format='%+2.0f dB')
plt.tight_layout(); plt.show()

## Livrable semaine 1

Si la dernière cellule affiche deux spectrogrammes côte à côte, le livrable est atteint :
> `for noisy, clean in dataloader: ...` retourne des spectrogrammes valides.